# Verification: LCD Classification Accuracy (Table IV) and Ecosystem Results (Table VI)

This notebook reproduces two results from the paper: Table IV (classification fidelity per
category, from `human_validation.csv`) and Table VI (ecosystem-scale LCD results across the
full 309-repository validation corpus, from `cohort_data/`).

**Methodology note**: an earlier version of this package computed "validation" statistics by
comparing two machine-generated columns against each other -- this produced numbers that looked
like independent validation but were not. This notebook instead compares the classifier's own
output (`automated_classifier_category`) against `resolution_category`, which is the real,
human-reviewer-confirmed ground truth, built through a two-round process documented in the
paper's Ground-Truth Validation Methodology paragraph and in this package's README.

In [1]:
import pandas as pd
from sklearn.metrics import classification_report, cohen_kappa_score

df = pd.read_csv("human_validation.csv")
print(f"Loaded {len(df)} rows")
print(df["resolution_method"].value_counts())


Loaded 1935 rows
resolution_method
unanimous_human_agreement             1267
unanimous_reviewer_reannotation        560
auto_resolved_undefined_source_url      84
corrected_by_engine_audit               13
resolved_by_reannotation_principle       9
corrected_for_batch_c_consistency        1
resolved_by_batch_consistency            1
Name: count, dtype: int64


In [2]:
# Real classification report: automated classifier prediction vs. reviewer-resolved ground truth
# (Category 0 excluded -- it is a baseline "no match" case, not part of the 5-tier framework)
sub = df[df["resolution_category"].isin([1, 2, 3, 4, 5])].copy()
y_true = sub["resolution_category"].astype(int)
y_pred = sub["automated_classifier_category"].astype(int)

print(f"N = {len(sub)}")
print(classification_report(y_true, y_pred, labels=[1, 2, 3, 4, 5], digits=3, zero_division=0))
print("Cohen's kappa (automated vs. reviewer-resolved ground truth):", round(cohen_kappa_score(y_true, y_pred), 4))


N = 1934
              precision    recall  f1-score   support

           1      0.969     1.000     0.984       410
           2      1.000     0.965     0.982       400
           3      1.000     1.000     1.000       389
           4      0.754     1.000     0.860       257
           5      0.997     0.824     0.903       478

    accuracy                          0.949      1934
   macro avg      0.944     0.958     0.946      1934
weighted avg      0.960     0.949     0.950      1934

Cohen's kappa (automated vs. reviewer-resolved ground truth): 0.9365


In [3]:
# Round-1 inter-reviewer reliability (binary accept/reject task that flagged Category 4 for
# investigation -- see the paper's Ground-Truth Validation Methodology paragraph for why this
# number alone is not sufficient as a validation metric, and why round 2 was necessary)
round1 = df[df["resolution_method"] == "unanimous_human_agreement"]
print(f"Round-1 unanimous agreement: {len(round1)} / {len(df)} rows")
# Note: reviewer_agreement_1/2 hold the original binary Agreed/Not Agreed judgments for ALL rows,
# including those later re-examined in round 2.
y1 = df["reviewer_agreement_1"].astype(str).str.strip()
y2 = df["reviewer_agreement_2"].astype(str).str.strip()
print("Round-1 Cohen's kappa (binary accept/reject):", round(cohen_kappa_score(y1, y2), 4))


Round-1 unanimous agreement: 1267 / 1935 rows
Round-1 Cohen's kappa (binary accept/reject): 0.8372


In [4]:
# Resolution-method breakdown -- shows how each row's final category was determined
print(df["resolution_method"].value_counts())
print()
print("Rows where resolution_category differs from the classifier's original output:")
changed = df["automated_classifier_category"] != df["resolution_category"]
print(f"{changed.sum()} / {len(df)} ({changed.mean()*100:.1f}%)")
print(df.loc[changed].groupby(["automated_classifier_category", "resolution_category"]).size())


resolution_method
unanimous_human_agreement             1267
unanimous_reviewer_reannotation        560
auto_resolved_undefined_source_url      84
corrected_by_engine_audit               13
resolved_by_reannotation_principle       9
corrected_for_batch_c_consistency        1
resolved_by_batch_consistency            1
Name: count, dtype: int64

Rows where resolution_category differs from the classifier's original output:
98 / 1935 (5.1%)
automated_classifier_category  resolution_category
1                              2.0                    13
4                              5.0                    84
5                              2.0                     1
dtype: int64


# Verification: Ecosystem Feasibility Results (Table VI)

This section reproduces Table VI of the paper -- the per-cohort method-level LCD classification
results across the full validation corpus (309 repositories, 107,208 methods) -- directly from
`cohort_data/{corporate,foundation,individual}_validation_source_sink.csv`. Repository counts per
cohort are recovered as `sink_project_id.nunique()`: each raw per-repository mining file
corresponds to exactly one scanned ("query") project, so this count is independent of the
method-level row count.

In [5]:
cohort_files = {
    "Corporate": "cohort_data/corporate_validation_source_sink.csv",
    "OSS Fdn.": "cohort_data/foundation_validation_source_sink.csv",
    "Individual": "cohort_data/individual_validation_source_sink.csv",
}

rows = []
grand_cat = {str(i): 0 for i in range(6)}
grand_total = 0
grand_projs = 0

for label, path in cohort_files.items():
    cdf = pd.read_csv(path, dtype=str)
    n_proj = cdf["sink_project_id"].nunique()
    n_methods = len(cdf)
    cat_counts = cdf["automated_classifier_category"].value_counts().to_dict()
    cat1, cat2, cat3, cat4, cat5, cat0 = (cat_counts.get(str(i), 0) for i in [1, 2, 3, 4, 5, 0])
    lcd_density = (cat3 + cat4) / (n_methods - cat0) * 100
    rows.append((label, n_proj, n_methods, cat1, cat2, cat3, cat4, cat5, cat0, lcd_density))
    grand_projs += n_proj
    grand_total += n_methods
    for i in range(6):
        grand_cat[str(i)] += cat_counts.get(str(i), 0)

print(f"{'Cohort':<12}{'Projs':>7}{'Methods':>10}{'Category 1':>16}{'Category 2':>16}{'Category 3':>16}{'Category 4':>15}{'Category 5':>18}{'Category 0':>18}{'LCD_density':>13}")
for label, n_proj, n_methods, cat1, cat2, cat3, cat4, cat5, cat0, lcd in rows:
    print(f"{label:<12}{n_proj:>7}{n_methods:>10,}"
          f"{cat1:>10,} ({cat1/n_methods*100:4.1f}%)"
          f"{cat2:>10,} ({cat2/n_methods*100:4.1f}%)"
          f"{cat3:>10,} ({cat3/n_methods*100:4.1f}%)"
          f"{cat4:>9,} ({cat4/n_methods*100:4.1f}%)"
          f"{cat5:>12,} ({cat5/n_methods*100:4.1f}%)"
          f"{cat0:>12,} ({cat0/n_methods*100:4.1f}%)"
          f"{lcd:>9.1f}%")

cat1, cat2, cat3, cat4, cat5, cat0 = (grand_cat[str(i)] for i in [1, 2, 3, 4, 5, 0])
lcd_total = (cat3 + cat4) / (grand_total - cat0) * 100
print(f"\n{'TOTAL':<12}{grand_projs:>7}{grand_total:>10,}"
      f"{cat1:>10,} ({cat1/grand_total*100:4.1f}%)"
      f"{cat2:>10,} ({cat2/grand_total*100:4.1f}%)"
      f"{cat3:>10,} ({cat3/grand_total*100:4.1f}%)"
      f"{cat4:>9,} ({cat4/grand_total*100:4.1f}%)"
      f"{cat5:>12,} ({cat5/grand_total*100:4.1f}%)"
      f"{cat0:>12,} ({cat0/grand_total*100:4.1f}%)"
      f"{lcd_total:>9.1f}%")

Cohort        Projs   Methods      Category 1      Category 2      Category 3     Category 4        Category 5        Category 0  LCD_density
Corporate        98    45,472     9,804 (21.6%)     2,846 ( 6.3%)     4,890 (10.8%)    1,006 ( 2.2%)      13,802 (30.4%)      13,124 (28.9%)     18.2%
OSS Fdn.        117    34,852     3,285 ( 9.4%)     1,351 ( 3.9%)     4,786 (13.7%)      410 ( 1.2%)      18,410 (52.8%)       6,610 (19.0%)     18.4%
Individual       94    26,884     2,812 (10.5%)     2,549 ( 9.5%)     3,447 (12.8%)      392 ( 1.5%)      12,592 (46.8%)       5,092 (18.9%)     17.6%

TOTAL           309   107,208    15,901 (14.8%)     6,746 ( 6.3%)    13,123 (12.2%)    1,808 ( 1.7%)      44,804 (41.8%)      24,826 (23.2%)     18.1%


# Verification: DSR Engine Self-Consistency Check

This section re-runs `DSR_engine.py`'s own `classify_violation()` against every non-Category-0
pair in the full 309-repository corpus (`cohort_data/`) and compares its output to the
`automated_classifier_category` already recorded in those files. A mismatch would mean the
recorded category was not actually produced by the shipped engine; this reproduces the mismatch
rate cited in the paper's replication package documentation.

In [6]:
from DSR_engine import classify_violation, LICENSE_LIST, license_mapping

full = pd.concat([pd.read_csv(path, dtype=str) for path in cohort_files.values()], ignore_index=True)
non_cat0 = full[full["automated_classifier_category"] != "0"].copy()


def normalize(license_str):
    if pd.isna(license_str):
        return "Unknown"
    return license_mapping.get(license_str.strip(), license_str.strip())


mismatches = 0
for _, row in non_cat0.iterrows():
    l_orig = normalize(row["source_license"])
    l_sink = normalize(row["sink_license"])
    predicted_cat, _ = classify_violation(l_orig, l_sink, LICENSE_LIST)
    if predicted_cat != row["automated_classifier_category"]:
        mismatches += 1

print(f"Non-Category-0 pairs checked: {len(non_cat0)}")
print(f"Mismatches between classify_violation() and the recorded category: {mismatches}")
print(f"Mismatch rate: {mismatches / len(non_cat0) * 100:.3f}%")

Non-Category-0 pairs checked: 82382
Mismatches between classify_violation() and the recorded category: 0
Mismatch rate: 0.000%
